##### Copyright 2025 Google LLC.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.


# vrp_breaks_from_start

<table align="left">
<td>
<a href="https://colab.research.google.com/github/google/or-tools/blob/main/examples/notebook/routing/vrp_breaks_from_start.ipynb"><img src="https://raw.githubusercontent.com/google/or-tools/main/tools/colab_32px.png"/>Run in Google Colab</a>
</td>
<td>
<a href="https://github.com/google/or-tools/blob/main/ortools/routing/samples/vrp_breaks_from_start.py"><img src="https://raw.githubusercontent.com/google/or-tools/main/tools/github_32px.png"/>View source on GitHub</a>
</td>
</table>

First, you must install [ortools](https://pypi.org/project/ortools/) package in this colab.

In [ ]:
%pip install ortools


Vehicles Routing Problem (VRP) with breaks relative to the vehicle start time.

Each vehicles start at T:15min, T:30min, T:45min and T:60min respectively.

Each vehicle must perform a break lasting 5 minutes,
starting between 25 and 45 minutes after route start.
e.g. vehicle 2 starting a T:45min must start a 5min breaks
between [45+25,45+45] i.e. in the range [70, 90].

Durations are in minutes.



In [ ]:
from typing import Any, Dict

from ortools.constraint_solver.python import constraint_solver
from ortools.routing import enums_pb2, parameters_pb2
from ortools.routing.python import routing



def create_data_model() -> Dict[str, Any]:
    """Stores the data for the problem."""
    data = {}
    data["num_vehicles"] = 4
    data["depot"] = 0
    data["time_matrix"] = [
        [0, 27, 38, 34, 29, 13, 25, 9, 15, 9, 26, 25, 19, 17, 23, 38, 33],
        [27, 0, 34, 15, 9, 25, 36, 17, 34, 37, 54, 29, 24, 33, 50, 43, 60],
        [38, 34, 0, 49, 43, 25, 13, 40, 23, 37, 20, 63, 58, 56, 39, 77, 37],
        [34, 15, 49, 0, 5, 32, 43, 25, 42, 44, 61, 25, 31, 41, 58, 28, 67],
        [29, 9, 43, 5, 0, 26, 38, 19, 36, 38, 55, 20, 25, 35, 52, 33, 62],
        [13, 25, 25, 32, 26, 0, 11, 15, 9, 12, 29, 38, 33, 31, 25, 52, 35],
        [25, 36, 13, 43, 38, 11, 0, 26, 9, 23, 17, 50, 44, 42, 25, 63, 24],
        [9, 17, 40, 25, 19, 15, 26, 0, 17, 19, 36, 23, 17, 16, 33, 37, 42],
        [15, 34, 23, 42, 36, 9, 9, 17, 0, 13, 19, 40, 34, 33, 16, 54, 25],
        [9, 37, 37, 44, 38, 12, 23, 19, 13, 0, 17, 26, 21, 19, 13, 40, 23],
        [26, 54, 20, 61, 55, 29, 17, 36, 19, 17, 0, 43, 38, 36, 19, 57, 17],
        [25, 29, 63, 25, 20, 38, 50, 23, 40, 26, 43, 0, 5, 15, 32, 13, 42],
        [19, 24, 58, 31, 25, 33, 44, 17, 34, 21, 38, 5, 0, 9, 26, 19, 36],
        [17, 33, 56, 41, 35, 31, 42, 16, 33, 19, 36, 15, 9, 0, 17, 21, 26],
        [23, 50, 39, 58, 52, 25, 25, 33, 16, 13, 19, 32, 26, 17, 0, 38, 9],
        [38, 43, 77, 28, 33, 52, 63, 37, 54, 40, 57, 13, 19, 21, 38, 0, 39],
        [33, 60, 37, 67, 62, 35, 24, 42, 25, 23, 17, 42, 36, 26, 9, 39, 0],
    ]
    # 15 min of service time
    data["service_time"] = [15] * len(data["time_matrix"])
    data["service_time"][data["depot"]] = 0
    assert len(data["time_matrix"]) == len(data["service_time"])
    return data


def print_solution(
    manager: routing.IndexManager,
    routing_model: routing.Model,
    solution: constraint_solver.Assignment,
) -> None:
    """Prints solution on console."""
    print(f"Objective: {solution.objective_value()}")

    print("Breaks:")
    intervals = solution.interval_var_container()
    for i in range(intervals.size()):
        brk = intervals.element(i)
        if brk.performed_value() == 1:
            print(
                f"{brk.var().name}: "
                + f"Start({brk.start_value()}) Duration({brk.duration_value()})"
            )
        else:
            print(f"{brk.var().name}: Unperformed")

    time_dimension = routing_model.get_dimension_or_die("Time")
    total_time = 0
    start_time = 0
    for vehicle_id in range(manager.num_vehicles()):
        if not routing_model.is_vehicle_used(solution, vehicle_id):
            continue
        index = routing_model.start(vehicle_id)
        plan_output = f"Route for vehicle {vehicle_id}:\n"
        while not routing_model.is_end(index):
            time_var = time_dimension.cumul_var(index)
            if routing_model.is_start(index):
                start_time = solution.value(time_var)
            plan_output += f"{manager.index_to_node(index)} "
            plan_output += f"Time({solution.value(time_var)}) -> "
            index = solution.value(routing_model.next_var(index))
        time_var = time_dimension.cumul_var(index)
        plan_output += f"{manager.index_to_node(index)} "
        plan_output += f"Time({solution.value(time_var)})"
        print(plan_output)
        route_time = solution.value(time_var) - start_time
        print(f"Time of the route: {route_time}min\n")
        total_time += route_time
    print(f"Total time of all routes: {total_time}min")


def main() -> None:
    """Solve the VRP with time windows."""
    # Instantiate the data problem.
    data = create_data_model()

    # Create the routing index manager.
    manager = routing.IndexManager(
        len(data["time_matrix"]), data["num_vehicles"], data["depot"]
    )

    # Create Routing Model.
    routing_model = routing.Model(manager)

    # Create and register a transit callback.
    def time_callback(from_index: int, to_index: int) -> int:
        """Returns the travel time between the two nodes."""
        # Convert from routing variable Index to time matrix NodeIndex.
        from_node = manager.index_to_node(from_index)
        to_node = manager.index_to_node(to_index)
        return data["time_matrix"][from_node][to_node]

    transit_callback_index = routing_model.register_transit_callback(time_callback)

    # Define cost of each arc.
    routing_model.set_arc_cost_evaluator_of_all_vehicles(transit_callback_index)

    # Add Time Windows constraint.
    time = "Time"
    routing_model.add_dimension(
        transit_callback_index,
        10,  # need optional waiting time to place break
        180,  # maximum time per vehicle
        False,  # Don't force start cumul to zero.
        time,
    )
    time_dimension = routing_model.get_dimension_or_die(time)
    time_dimension.set_global_span_cost_coefficient(10)

    # Each vehicle start with a 15min delay
    for vehicle_id in range(manager.num_vehicles()):
        index = routing_model.start(vehicle_id)
        time_dimension.cumul_var(index).set_value((vehicle_id + 1) * 15)

    # Add breaks
    # warning: Need a pre-travel array using the solver's index order.
    node_visit_transit = [0] * routing_model.size()
    for index in range(routing_model.size()):
        node = manager.index_to_node(index)
        node_visit_transit[index] = data["service_time"][node]

    # Add a break lasting 5 minutes, start between 25 and 45 minutes after route
    # start
    for v in range(manager.num_vehicles()):
        start_var = time_dimension.cumul_var(routing_model.start(v))
        break_start = (
            routing_model.solver.new_int_var(25, 45, f"break_start_window_{v}")
            + start_var
        )

        break_intervals = [
            routing_model.solver.new_fixed_duration_interval_var(
                break_start, 5, f"Break for vehicle {v}"
            )
        ]
        time_dimension.set_break_intervals_of_vehicle(
            break_intervals, v, node_visit_transit
        )

    # Setting first solution heuristic.
    search_parameters: parameters_pb2.RoutingSearchParameters = (
        routing.default_routing_search_parameters()
    )
    search_parameters.first_solution_strategy = (
        enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    )
    search_parameters.local_search_metaheuristic = (
        enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    )
    # search_parameters.log_search = True
    search_parameters.time_limit.seconds = 2

    # Solve the problem.
    solution = routing_model.solve_with_parameters(search_parameters)

    # Print solution on console.
    if solution:
        print_solution(manager, routing_model, solution)
    else:
        print("No solution found !")


main()

